In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import numpy as np
from PIL import Image
import os
from glob import glob

# -------------------------
# 1. Custom Dataset
# -------------------------
class ObjectDetectionDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        """
        images_dir: folder with images
        masks_dir: folder with binary masks for objects
        """
        self.image_paths = sorted(glob(os.path.join(images_dir, "*.png")))
        self.mask_paths = sorted(glob(os.path.join(masks_dir, "*.png")))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def get_bbox(self, mask):
        """Convert binary mask to bounding box [x_min, y_min, x_max, y_max]"""
        coords = np.argwhere(mask)
        if coords.shape[0] == 0:
            return None
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        return [x_min, y_min, x_max, y_max]

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        mask = np.array(Image.open(self.mask_paths[idx]))  
        mask = (mask > 127).astype(np.uint8)

        bbox = self.get_bbox(mask)
        if bbox is None:
            bbox = [0, 0, 1, 1]  # fallback

        # Convert to tensor
        img_tensor = F.to_tensor(img)
        bbox_tensor = torch.tensor([bbox], dtype=torch.float32)
        labels_tensor = torch.tensor([1], dtype=torch.int64)  # 1 class

        target = {"boxes": bbox_tensor, "labels": labels_tensor}

        if self.transform:
            img_tensor, target = self.transform(img_tensor, target)

        return img_tensor, target

# -------------------------
# 2. Model
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = fasterrcnn_resnet50_fpn(pretrained=True)

num_classes = 2  # background + 1 object
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features, num_classes
)

model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# -------------------------
# 3. Dataset & DataLoader
# -------------------------
images_dir = "data/images"
masks_dir = "data/masks"

dataset = ObjectDetectionDataset(images_dir, masks_dir)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

# -------------------------
# 4. Training Loop
# -------------------------
num_epochs = 5
model.train()

for epoch in range(num_epochs):
    epoch_loss = 0
    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss/len(dataloader):.4f}")

C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\Dell/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100.0%


Epoch 1, Loss: 0.3160
Epoch 2, Loss: 0.1040
Epoch 3, Loss: 0.0713
Epoch 4, Loss: 0.0579
Epoch 5, Loss: 0.0753
